# NB1 — Chỉ số kỹ thuật → dự báo xu hướng (uptrend / sideway / downtrend)

Nhánh **kỹ thuật** của luồng nghiên cứu. Đầu vào duy nhất là tệp **OHLCV vàng**
ở khung bất kỳ (M1 … W1). Đầu ra `gold_price_technical_signal.csv` gồm các chỉ
báo và 4 dự báo xu hướng theo luật: MA, RSI, MACD và dự báo tổng hợp (≥ 2/3 luật
đồng ý). Cột `Trend` ghi dạng chữ UPTREND / SIDEWAY / DOWNTREND.

```
                     OHLCV vàng (khung bất kỳ)
                    /                          \
   NB1 chỉ báo kỹ thuật                    NB2 3 model AI
   luật → dự báo xu hướng                  XGBoost, RF, Bi-LSTM → dự báo xu hướng
                    \                          /
        NB3 chiến lược: xu hướng → lệnh BUY / SELL / FLAT
            → backtest trên CÙNG giai đoạn → Profit, Sharpe, Max DD…
```

NB1 và NB2 chỉ **dự báo xu hướng**: **1 = uptrend (trend tăng) · 0 = sideway ·
−1 = downtrend (trend giảm)** — chưa phải lệnh giao dịch. **NB3** mới áp chiến lược để
đổi xu hướng thành lệnh **BUY / SELL / FLAT** rồi backtest.

NB1 và NB2 **độc lập với nhau**: mỗi notebook tự đọc tệp OHLCV, chạy trước hay sau
đều được. NB3 chạy sau cùng, khi đã có tệp kết quả của cả hai.

Import thư viện và nơi lưu tệp

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Nơi trao đổi tệp giữa 3 notebook ─────────────────────────────
# Mỗi notebook Colab chạy trên một máy ảo riêng: tệp NB1/NB2 tạo ra KHÔNG tự có
# mặt ở NB3. Vì vậy cả 3 notebook cùng đọc/ghi vào MỘT thư mục trên Google Drive.
THU_MUC_DRIVE = '/content/drive/MyDrive/Data_NghienCuu'

try:
    from google.colab import files, drive
    TREN_COLAB = True
except ImportError:
    files = drive = None
    TREN_COLAB = False

THU_MUC = '.'
if TREN_COLAB:
    try:
        drive.mount('/content/drive')
        THU_MUC = THU_MUC_DRIVE
    except Exception as loi:
        print('Không gắn được Google Drive (%s).' % loi)
        print('→ Dùng /content: nhớ tải tệp kết quả về và tải lên ở NB3.')
        THU_MUC = '/content'
os.makedirs(THU_MUC, exist_ok=True)
print('Thư mục trao đổi dữ liệu:', os.path.abspath(THU_MUC))


def tim_tep(ten):
    """Tìm tệp đầu vào: thư mục trao đổi → thư mục hiện tại → tải lên (Colab)."""
    for p in (os.path.join(THU_MUC, ten), ten):
        if os.path.exists(p):
            return p
    if TREN_COLAB:
        print('Chưa thấy %s trong %s — hãy tải tệp này lên:' % (ten, THU_MUC))
        up = files.upload()
        if up:
            return list(up.keys())[0]
    raise FileNotFoundError('Không tìm thấy %s. Hãy chạy notebook tạo ra tệp này trước.' % ten)


def luu_tep(bang, ten):
    p = os.path.join(THU_MUC, ten)
    bang.to_csv(p, index=False, encoding='utf-8-sig')
    print('Đã lưu: %s  (%d dòng × %d cột)' % (os.path.abspath(p), bang.shape[0], bang.shape[1]))
    return p

Nhập bộ dữ liệu OHLCV

Để trống `DUONG_DAN_DU_LIEU` thì Colab hiện nút **Choose Files** để tải tệp lên.
Để khỏi tải cùng một tệp hai lần cho NB1 và NB2, có thể đặt tệp vào Drive rồi
điền đường dẫn, ví dụ `/content/drive/MyDrive/Data_NghienCuu/xau_h1.csv`.
Nhận `.csv`, `.txt` (tách bằng dấu phẩy, `;` hoặc tab), `.xlsx`, `.parquet`.

In [ ]:
DUONG_DAN_DU_LIEU = ''

DUONG_DAN_DU_LIEU = DUONG_DAN_DU_LIEU or os.environ.get('NCKH_DU_LIEU', '')
if DUONG_DAN_DU_LIEU:
    file_name = DUONG_DAN_DU_LIEU
elif TREN_COLAB:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
else:
    raise ValueError('Đang chạy ngoài Colab: hãy điền DUONG_DAN_DU_LIEU.')

print("Tệp dữ liệu:", file_name)

Đọc dữ liệu

In [ ]:
ten_thuong = file_name.lower()
if ten_thuong.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
elif ten_thuong.endswith(('.parquet', '.pq')):
    df = pd.read_parquet(file_name)
else:
    df = pd.read_csv(file_name)
    # Tệp xuất từ MetaTrader thường tách cột bằng tab hoặc ';' → tự dò lại
    if df.shape[1] == 1:
        df = pd.read_csv(file_name, sep=None, engine='python')

print("Kích thước dữ liệu:", df.shape)
display(df.head())

 PHẦN A — XỬ LÝ DỮ LIỆU

In [ ]:
print("Các cột trong dữ liệu:")
print(df.columns.tolist())

Chuẩn hóa tên cột và cột thời gian (dùng được cho mọi khung)

Dữ liệu vàng từ các nguồn khác nhau đặt tên cột rất khác nhau. Ô dưới tự nhận
biết mà không cần sửa tay:

- **Tên cột thời gian:** `time`, `datetime`, `date`, `timestamp`, `Gmt time`,
  `<DATE>` + `<TIME>` tách rời (kiểu MetaTrader)…
- **Định dạng thời gian:** `2025-01-02 13:00`, `2025.01.02 13:00`, `02/01/2025`,
  số giây hoặc mili-giây Unix, `20250102`, có hoặc không có múi giờ.
- **Tên cột giá:** `Open/open/<OPEN>/o`, `Close/Adj Close/price`,
  `Volume/Tick Volume/tickvol`…
- **Định dạng số:** `1183.949`, `1183,949`, `1,183.949`, `1.183,949`.

Mọi thời điểm được đưa về **UTC, không kèm múi giờ**. Khung thời gian được suy
ra từ khoảng cách phổ biến nhất giữa hai nến liền nhau.

Ô này **giống hệt nhau ở NB1 và NB2**, nên hai notebook luôn đọc cùng một tệp ra
cùng một bảng dữ liệu.

In [ ]:
def _chuan_ten(c):
    return ' '.join(str(c).strip().lower().replace('<', ' ').replace('>', ' ').replace('_', ' ').split())

# Tên cột theo thứ tự ưu tiên
BI_DANH = {
    'time':   ['datetime', 'date time', 'timestamp', 'time', 'date', 'gmt time', 'local time',
               'time (utc)', 'datetime utc', 'open time', 'opentime', 'thoi gian', 'ngay'],
    'open':   ['open', 'o', 'open price', 'gia mo'],
    'high':   ['high', 'h', 'high price', 'gia cao'],
    'low':    ['low', 'l', 'low price', 'gia thap'],
    'close':  ['close', 'c', 'close price', 'adj close', 'price', 'last', 'gia dong'],
    'volume': ['volume', 'vol', 'tick volume', 'tickvol', 'real volume', 'khoi luong'],
}

def _tim_cot(cac_cot, loai):
    ten = {_chuan_ten(x): x for x in cac_cot}
    for ung_vien in BI_DANH[loai]:
        if ung_vien in ten:
            return ten[ung_vien]
    return None

def _giong_gio(s):
    """Cột chỉ chứa giờ dạng 13:00 hoặc 13:00:00 (cột <TIME> tách rời)."""
    v = s.dropna().astype(str).str.strip().head(200)
    return len(v) > 0 and v.str.fullmatch(r'\d{1,2}:\d{2}(:\d{2})?').mean() > 0.9

def _so(s):
    """Số thực. Chấp nhận 1183.949 · 1183,949 · 1,183.949 · 1.183,949."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    s = s.astype(str).str.strip().str.replace(' ', '', regex=False)
    mau = s.head(500)
    # Dấu nào đứng SAU CÙNG là dấu thập phân; dấu còn lại là phân cách hàng nghìn
    phay_la_thap_phan = (mau.str.rfind(',') > mau.str.rfind('.')).mean() > 0.5
    if phay_la_thap_phan:
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    else:
        s = s.str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors='coerce')

def _doc_thoi_gian(s):
    """Đọc cột thời gian ở mọi định dạng thường gặp, trả về UTC không múi giờ."""
    if pd.api.types.is_numeric_dtype(s):
        v = pd.to_numeric(s, errors='coerce')
        m = v.dropna().abs().median()
        if 1e7 <= m < 1e8:                                   # dạng 20250102
            return pd.to_datetime(v.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
        don_vi = 'ms' if m > 1e11 else 's'                   # Unix mili-giây hay giây
        return pd.to_datetime(v, unit=don_vi, errors='coerce', utc=True).dt.tz_localize(None)

    s = s.astype(str).str.strip()
    t = pd.to_datetime(s, errors='coerce', utc=True)
    if t.isna().mean() > 0.01:                               # định dạng lẫn lộn → đọc từng dòng
        t = pd.to_datetime(s, errors='coerce', utc=True, format='mixed')
    ung_vien = [t]
    if s.str.contains('/').mean() > 0.5:                     # 02/01/2025: ngày-trước hay tháng-trước?
        ung_vien.append(pd.to_datetime(s, errors='coerce', utc=True, format='mixed', dayfirst=True))
    # Dữ liệu giá luôn xếp theo thời gian: chọn cách đọc ít lỗi nhất và tăng dần nhiều nhất
    diem = lambda x: (x.notna().mean(), (x.diff().dt.total_seconds() > 0).mean())
    return max(ung_vien, key=diem).dt.tz_localize(None)

KHUNG_CHUAN = [(1, 'M1'), (5, 'M5'), (15, 'M15'), (30, 'M30'), (60, 'H1'),
               (240, 'H4'), (1440, 'D1'), (10080, 'W1'), (43200, 'MN')]

def nhan_dien_khung(t):
    phut = t.sort_values().diff().dt.total_seconds().div(60)
    buoc = phut[phut > 0].mode().iloc[0]
    return min(KHUNG_CHUAN, key=lambda k: abs(np.log(k[0] / buoc)))[1], buoc


# ── 1. Cột thời gian
cac_cot = list(df.columns)
ten_chuan = {_chuan_ten(x): x for x in cac_cot}
if 'date' in ten_chuan and 'time' in ten_chuan and _giong_gio(df[ten_chuan['time']]):
    cot_tg = '%s + %s' % (ten_chuan['date'], ten_chuan['time'])
    tho_tg = df[ten_chuan['date']].astype(str).str.strip() + ' ' + df[ten_chuan['time']].astype(str).str.strip()
else:
    cot_tg = _tim_cot(cac_cot, 'time')
    if cot_tg is None:
        raise ValueError('Không tìm thấy cột thời gian trong: %s' % cac_cot)
    tho_tg = df[cot_tg]

ra = pd.DataFrame({'Date': _doc_thoi_gian(tho_tg)})

# ── 2. Cột giá và khối lượng
anh_xa = {}
for loai, ten_moi in [('open', 'Open'), ('high', 'High'), ('low', 'Low'),
                      ('close', 'Close'), ('volume', 'Volume')]:
    cot = _tim_cot(cac_cot, loai)
    anh_xa[ten_moi] = cot
    ra[ten_moi] = _so(df[cot]).values if cot is not None else np.nan

if anh_xa['Close'] is None:
    raise ValueError('Không tìm thấy cột giá đóng cửa trong: %s' % cac_cot)
for ten_moi in ('Open', 'High', 'Low'):
    if anh_xa[ten_moi] is None:
        print('⚠ Thiếu cột %s → tạm dùng giá Close.' % ten_moi)
        ra[ten_moi] = ra['Close']
if anh_xa['Volume'] is None:
    ra['Volume'] = 1.0            # nhiều nguồn Forex không có khối lượng thật

df = ra
KHUNG, BUOC_PHUT = nhan_dien_khung(df['Date'].dropna())

print('Ánh xạ cột:')
print('  %-7s ← %s' % ('Date', cot_tg))
for k, v in anh_xa.items():
    print('  %-7s ← %s' % (k, v if v is not None else '(không có)'))
print('\nKhung thời gian nhận diện: %s  (bước phổ biến %.0f phút)' % (KHUNG, BUOC_PHUT))
print('Giai đoạn: %s → %s' % (df['Date'].min(), df['Date'].max()))
print('Dòng không đọc được thời gian: %d' % df['Date'].isna().sum())

Loại bỏ dữ liệu lỗi và trùng

In [ ]:
print("Trước xử lý:", df.shape)

# Bỏ dòng thiếu thời gian hoặc giá đóng cửa
df = df.dropna(subset=['Date', 'Close'])

# Giá phải lớn hơn 0
df = df[df['Close'] > 0]

# Bỏ nến vi phạm hình học: High < Low, hoặc Close nằm ngoài [Low, High]
hop_le = (df['High'] >= df['Low']) & (df['Close'] <= df['High']) & (df['Close'] >= df['Low'])
print("Nến vi phạm hình học OHLC:", int((~hop_le).sum()))
df = df[hop_le]

# Xóa dòng trùng thời gian, sắp xếp theo thời gian
df = (
    df
    .drop_duplicates(subset=['Date'], keep='last')
    .sort_values('Date')
    .reset_index(drop=True)
)

print("Sau xử lý:", df.shape)
if len(df) < 100:
    raise ValueError('Sau khi làm sạch chỉ còn %d dòng. Kiểm tra lại ánh xạ cột ở ô trên '
                     'và định dạng số của tệp (dấu thập phân, dấu phân cách hàng nghìn).' % len(df))

Kiểm tra dữ liệu sau xử lý

In [ ]:
print("Số giá trị thiếu:")
display(df.isnull().sum())

print("\nSố dòng trùng:")
print(df.duplicated().sum())

print("\n5 dòng đầu:")
display(df.head())

print("\n5 dòng cuối:")
display(df.tail())

PHẦN B — XÂY DỰNG CHỈ BÁO KỸ THUẬT



return

In [ ]:
df['Return'] = np.log(
    df['Close'] /
    df['Close'].shift(1)
)

display(
    df[['Date', 'Close', 'Return']].head(10)
)

MA10, MA30, MA50


In [ ]:
df['MA10'] = (
    df['Close']
    .rolling(window=10)
    .mean()
)

df['MA30'] = (
    df['Close']
    .rolling(window=30)
    .mean()
)

df['MA50'] = (
    df['Close']
    .rolling(window=50)
    .mean()
)

display(
    df[
        ['Date', 'Close', 'MA10', 'MA30', 'MA50']
    ].tail(10)
)

EMA12, EMA26

In [ ]:
df['EMA12'] = (
    df['Close']
    .ewm(
        span=12,
        adjust=False
    )
    .mean()
)

df['EMA26'] = (
    df['Close']
    .ewm(
        span=26,
        adjust=False
    )
    .mean()
)

display(
    df[
        ['Date', 'Close', 'EMA12', 'EMA26']
    ].tail(10)
)

RSI14

In [ ]:
delta = df['Close'].diff()

gain = delta.clip(lower=0)

loss = -delta.clip(upper=0)

avg_gain = (
    gain
    .rolling(window=14)
    .mean()
)

avg_loss = (
    loss
    .rolling(window=14)
    .mean()
)

RS = avg_gain / avg_loss

df['RSI14'] = (
    100 -
    (
        100 / (1 + RS)
    )
)

display(
    df[
        ['Date', 'Close', 'RSI14']
    ].tail(10)
)

MACD


In [ ]:
df['MACD'] = (
    df['EMA12'] -
    df['EMA26']
)

df['MACD_Signal'] = (
    df['MACD']
    .ewm(
        span=9,
        adjust=False
    )
    .mean()
)

df['MACD_Hist'] = (
    df['MACD'] -
    df['MACD_Signal']
)

display(
    df[
        [
            'Date',
            'MACD',
            'MACD_Signal',
            'MACD_Hist'
        ]
    ].tail(10)
)

Bollinger Bands

In [ ]:
df['MA20'] = (
    df['Close']
    .rolling(window=20)
    .mean()
)

df['STD20'] = (
    df['Close']
    .rolling(window=20)
    .std()
)

df['BB_Upper'] = (
    df['MA20'] +
    2 * df['STD20']
)

df['BB_Lower'] = (
    df['MA20'] -
    2 * df['STD20']
)

display(
    df[
        [
            'Date',
            'Close',
            'MA20',
            'BB_Upper',
            'BB_Lower'
        ]
    ].tail(10)
)

Volatility20

In [ ]:
df['Volatility20'] = (
    df['Return']
    .rolling(window=20)
    .std()
)

display(
    df[
        [
            'Date',
            'Return',
            'Volatility20'
        ]
    ].tail(10)
)

PHẦN C — TẠO 3 TÍN HIỆU

Tín hiệu MA

In [ ]:
df['MA_Signal'] = 0

# Dự báo TREND TĂNG (1)
df.loc[
    df['MA10'] > df['MA30'],
    'MA_Signal'
] = 1

# Dự báo TREND GIẢM (-1)
df.loc[
    df['MA10'] < df['MA30'],
    'MA_Signal'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'MA10',
            'MA30',
            'MA_Signal'
        ]
    ].tail(20)
)

Tín hiệu RSI

In [ ]:
df['RSI_Signal'] = 0

# Dự báo TREND TĂNG (1)
df.loc[
    df['RSI14'] < 30,
    'RSI_Signal'
] = 1

# Dự báo TREND GIẢM (-1)
df.loc[
    df['RSI14'] > 70,
    'RSI_Signal'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'RSI14',
            'RSI_Signal'
        ]
    ].tail(20)
)

Tín hiệu MACD

In [ ]:
df['MACD_Diff'] = (
    df['MACD'] -
    df['MACD_Signal']
)

df['MACD_Signal_Final'] = 0


# MACD cắt lên
bullish_cross = (
    (df['MACD_Diff'] > 0) &
    (df['MACD_Diff'].shift(1) <= 0)
)


# MACD cắt xuống
bearish_cross = (
    (df['MACD_Diff'] < 0) &
    (df['MACD_Diff'].shift(1) >= 0)
)


df.loc[
    bullish_cross,
    'MACD_Signal_Final'
] = 1


df.loc[
    bearish_cross,
    'MACD_Signal_Final'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'MACD',
            'MACD_Signal',
            'MACD_Signal_Final'
        ]
    ].tail(30)
)

PHẦN D — TỔNG HỢP 3 TÍN HIỆU

Đếm số luật dự báo TREND TĂNG và TREND GIẢM

In [ ]:
df['Up_Count'] = (
    (df['MA_Signal'] == 1).astype(int)
    +
    (df['RSI_Signal'] == 1).astype(int)
    +
    (df['MACD_Signal_Final'] == 1).astype(int)
)


df['Down_Count'] = (
    (df['MA_Signal'] == -1).astype(int)
    +
    (df['RSI_Signal'] == -1).astype(int)
    +
    (df['MACD_Signal_Final'] == -1).astype(int)
)

Tạo kết quả cuối cùng -1 / 0 / 1

In [ ]:
df['Signal'] = 0


# Có ít nhất 2 luật dự báo trend tăng → UPTREND
df.loc[
    df['Up_Count'] >= 2,
    'Signal'
] = 1


# Có ít nhất 2 luật dự báo trend giảm → DOWNTREND
df.loc[
    df['Down_Count'] >= 2,
    'Signal'
] = -1

Kiểm tra kết quả

In [ ]:
display(
    df[
        [
            'Date',
            'Close',

            'MA_Signal',
            'RSI_Signal',
            'MACD_Signal_Final',

            'Up_Count',
            'Down_Count',

            'Signal'
        ]
    ].tail(50)
)

PHẦN E — ĐỔI RA UPTREND / SIDEWAY / DOWNTREND ĐỂ DỄ ĐỌC

Đây là **dự báo xu hướng**, chưa phải lệnh. NB3 mới đổi xu hướng thành lệnh BUY / SELL / FLAT theo chiến lược.


In [ ]:
df['Trend'] = df['Signal'].map({
    -1: 'DOWNTREND',
     0: 'SIDEWAY',
     1: 'UPTREND'
})

display(
    df[
        [
            'Date',
            'Close',
            'Signal',
            'Trend'
        ]
    ].tail(30)
)

PHẦN F — THỐNG KÊ KẾT QUẢ

In [ ]:
print("Số nến theo từng xu hướng dự báo:")

print(
    df['Trend'].value_counts()
)

Tỷ lệ UPTREND / SIDEWAY / DOWNTREND


In [ ]:
signal_percentage = (
    df['Trend']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Tỷ lệ xu hướng dự báo (%):")

print(signal_percentage)

PHẦN G — VẼ BIỂU ĐỒ

Giá và MA

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    df['Date'],
    df['Close'],
    label='Close'
)

plt.plot(
    df['Date'],
    df['MA10'],
    label='MA10'
)

plt.plot(
    df['Date'],
    df['MA30'],
    label='MA30'
)

plt.plot(
    df['Date'],
    df['MA50'],
    label='MA50'
)

plt.title('Gold Price and Moving Averages')

plt.xlabel('Date')
plt.ylabel('Price')

plt.legend()
plt.grid(True)

plt.show()

Bollinger Bands

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    df['Date'],
    df['Close'],
    label='Close'
)

plt.plot(
    df['Date'],
    df['BB_Upper'],
    label='Upper Band'
)

plt.plot(
    df['Date'],
    df['MA20'],
    label='MA20'
)

plt.plot(
    df['Date'],
    df['BB_Lower'],
    label='Lower Band'
)

plt.title('Bollinger Bands')

plt.xlabel('Date')
plt.ylabel('Price')

plt.legend()
plt.grid(True)

plt.show()

RSI


In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    df['Date'],
    df['RSI14'],
    label='RSI14'
)

plt.axhline(
    70,
    linestyle='--',
    label='Overbought 70'
)

plt.axhline(
    30,
    linestyle='--',
    label='Oversold 30'
)

plt.title('RSI(14)')

plt.xlabel('Date')
plt.ylabel('RSI')

plt.legend()
plt.grid(True)

plt.show()

PHẦN H — XUẤT FILE KẾT QUẢ

Đổi tên cột thời gian và giá về chữ thường (`time, open, high, low, close,
volume`) để NB3 đọc thống nhất với tệp của NB2.

In [ ]:
ket_qua = df.rename(columns={'Date': 'time', 'Open': 'open', 'High': 'high',
                             'Low': 'low', 'Close': 'close', 'Volume': 'volume'})

output_file = 'gold_price_technical_signal.csv'
duong_dan_ra = luu_tep(ket_qua, output_file)
print('Khung %s | %s → %s' % (KHUNG, ket_qua['time'].min(), ket_qua['time'].max()))

In [ ]:
# Tải tệp về máy (không bắt buộc nếu đã lưu trên Google Drive)
if TREN_COLAB and not duong_dan_ra.startswith('/content/drive'):
    files.download(duong_dan_ra)